# ⚙️ 05b — Feature Engineering v2
## Silver · Turbina Kelmarsh T1 · 2018–2021

---

### Qué cambia respecto a 05

El diagnóstico del `00_diagnostico.ipynb` reveló que **las features `slope_*` tienen AUC univariante = 0.500** en todas las familias — ruido puro.  
Esto se debe a que la degradación en turbinas eólicas no sigue una pendiente monótona; en cambio, la señal está en:
- **Percentiles extremos** (p95, p99): el sensor empieza a tocar valores que antes no alcanzaba
- **Tiempo acumulado sobre umbral**: cuántas horas en la última semana ha superado el p90 histórico
- **Ratio vs baseline**: el sensor actual comparado con su propio promedio de los primeros 6 meses (operación limpia)
- **Exceedance reciente**: número de veces que ha superado el umbral en ventana corta

Las features `mean_*` y `std_*` se mantienen — tienen AUC 0.53–0.67 y aportan señal real.  
Las `slope_*` se eliminan completamente.

| | 05 original | 05b nuevo |
|---|---|---|
| Stats por ventana | mean + std + slope | mean + std + p95 + exceedance |
| Features por sensor | 12 | 12 (misma cantidad, diferente contenido) |
| Baseline ratio | No | Sí (ratio vs media primeros 180 días) |
| Slope | Sí (AUC=0.50) | **Eliminado** |


## 1. Carga del Dataset Etiquetado

In [7]:
import os
import pandas as pd
import numpy as np

base_dir = os.path.dirname(os.getcwd())
silver_dir = os.path.join(base_dir, 'data', 'silver')

df = pd.read_parquet(os.path.join(silver_dir, 'dataset_labeled.parquet'))
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Dataset cargado: {len(df):,} filas x {len(df.columns)} columnas')
print(f'Rango: {df["timestamp"].min()} → {df["timestamp"].max()}')


Dataset cargado: 210,384 filas x 55 columnas
Rango: 2017-12-31 23:00:00 → 2021-12-31 22:50:00


---

## 2. Features Calculadas de Dominio

Idénticas a la versión 05 — estas tienen justificación física sólida y se mantienen todas.


In [8]:
# ==============================================================================
# 2. FEATURES CALCULADAS DE DOMINIO (sin cambios respecto a 05)
# ==============================================================================
original_columns = set(df.columns)

# --- YAW ---
df['yaw_error'] = (df['nacelle_position'] - df['wind_direction']).abs() % 360
df['yaw_error'] = df['yaw_error'].apply(lambda x: x if x <= 180 else 360 - x)
df['yaw_error_wind']   = df['yaw_error'] * df['wind_speed_ms']
df['cable_rate']       = df['cable_windings_from_calibration_point'].diff(periods=1).fillna(0)
df['nacelle_std_ratio']= df['nacelle_position_standard_deviation'] / (df['wind_speed_ms'] + 1e-6)

# --- TÉRMICAS GENERADOR ---
df['t_bearing_delta']      = df['generator_bearing_front_temperature_c'] - df['nacelle_ambient_temperature_c']
df['t_rear_bearing_delta'] = df['generator_bearing_rear_temperature_c']  - df['nacelle_ambient_temperature_c']
df['t_stator_delta']       = df['stator_temperature_1_c']                - df['nacelle_ambient_temperature_c']
df['t_gear_oil_delta']     = df['gear_oil_temperature_c']                - df['nacelle_ambient_temperature_c']
df['t_bearing_diff']       = df['generator_bearing_front_temperature_c'] - df['generator_bearing_rear_temperature_c']
df['t_stator_bearing_diff']= df['stator_temperature_1_c']                - df['generator_bearing_front_temperature_c']
df['t_bearing_delta_roc']  = df['t_bearing_delta'].diff(periods=6)
df['t_stator_roc']         = df['stator_temperature_1_c'].diff(periods=6)

# --- ELÉCTRICO ---
df['apparent_power_kva']  = (df['power_kw']**2 + df['reactive_power_kvar']**2) ** 0.5
df['reactive_power_ratio']= df['reactive_power_kvar'] / (df['apparent_power_kva'] + 1e-6)

# --- HIDRÁULICO ---
df['pressure_vs_temp']    = df['gear_oil_inlet_pressure_bar'] / (df['gear_oil_inlet_temperature_c'] + 273.15)
df['metal_particle_rate'] = df['metal_particle_count'].diff(periods=1).fillna(0).clip(lower=0)

# --- PITCH ---
df['pitch_asymmetry']        = (df[['blade_angle_pitch_position_a','blade_angle_pitch_position_b','blade_angle_pitch_position_c']].max(axis=1) -
                                 df[['blade_angle_pitch_position_a','blade_angle_pitch_position_b','blade_angle_pitch_position_c']].min(axis=1))
df['blade_angle_mean']       = df[['blade_angle_pitch_position_a','blade_angle_pitch_position_b','blade_angle_pitch_position_c']].mean(axis=1)
df['motor_current_imbalance']= df[['motor_current_axis_1_a','motor_current_axis_2_a','motor_current_axis_3_a']].std(axis=1)

calculated_features = [c for c in df.columns if c not in original_columns]
print(f'Features de dominio añadidas: {len(calculated_features)}')
for f in calculated_features:
    print(f'  {f}')


Features de dominio añadidas: 19
  yaw_error
  yaw_error_wind
  cable_rate
  nacelle_std_ratio
  t_bearing_delta
  t_rear_bearing_delta
  t_stator_delta
  t_gear_oil_delta
  t_bearing_diff
  t_stator_bearing_diff
  t_bearing_delta_roc
  t_stator_roc
  apparent_power_kva
  reactive_power_ratio
  pressure_vs_temp
  metal_particle_rate
  pitch_asymmetry
  blade_angle_mean
  motor_current_imbalance


---

## 3. Baseline de Operación Limpia (primeros 180 días)

Para cada sensor calculamos la **media y p90 de los primeros 180 días** de operación.  
Esto representa el comportamiento "sano" de la turbina antes de cualquier degradación apreciable.

El ratio `sensor_actual / baseline_medio` es la feature más potente para degradación:
- Ratio = 1.0 → comportamiento normal
- Ratio > 1.2 → sensor un 20% por encima de su baseline → señal de degradación

El p90 del baseline se usa como umbral para el `exceedance_count`.


In [9]:
# ==============================================================================
# 3. BASELINE DE OPERACIÓN LIMPIA
# ==============================================================================
BASELINE_DAYS = 180  # primeros 6 meses = operación limpia
baseline_cutoff = df['timestamp'].min() + pd.Timedelta(days=BASELINE_DAYS)

df_baseline = df[df['timestamp'] < baseline_cutoff]
print(f'Período baseline: {df["timestamp"].min().date()} → {baseline_cutoff.date()} ({len(df_baseline):,} filas)')

# Todos los sensores numéricos (excluir timestamps y targets)
all_sensor_cols = [c for c in df.columns 
                   if c not in ['timestamp'] 
                   and not c.startswith('is_pre_') 
                   and not c.startswith('hours_to_')
                   and df[c].dtype in [float, 'float64', 'float32']]

baseline_mean = df_baseline[all_sensor_cols].mean()
baseline_p90  = df_baseline[all_sensor_cols].quantile(0.90)
baseline_p10  = df_baseline[all_sensor_cols].quantile(0.10)

print(f'Baseline calculado para {len(all_sensor_cols)} sensores/features')
print(f'Ejemplo — cable_windings baseline mean: {baseline_mean.get("cable_windings_from_calibration_point", "N/A"):.2f}')


Período baseline: 2017-12-31 → 2018-06-29 (25,926 filas)
Baseline calculado para 64 sensores/features
Ejemplo — cable_windings baseline mean: -0.06


---

## 4. Función de Rolling Features v2

### Qué se calcula ahora en cada ventana

| Stat | v05 | v05b | Justificación |
|------|-----|------|---------------|
| `mean` | ✅ | ✅ | Nivel promedio — detecta derivas lentas |
| `std` | ✅ | ✅ | Variabilidad — detecta inestabilidad creciente |
| `slope` | ✅ | ❌ | **Eliminado** — AUC=0.50, ruido puro |
| `p95` | ❌ | ✅ | **Nuevo** — el sensor empieza a tocar valores extremos |
| `exceedance` | ❌ | ✅ | **Nuevo** — horas por encima del p90 baseline |

El `exceedance` es particularmente potente para degradación mecánica:  
un rodamiento degradado no calienta siempre más, sino que tiene **picos** cada vez más frecuentes.


In [10]:
# ==============================================================================
# 4. FUNCIÓN DE ROLLING FEATURES v2
# ==============================================================================
WINDOWS = {'1h': 6, '6h': 36, '24h': 144, '7d': 1008}

def make_rolling_features_v2(df, sensors, windows, baseline_mean, baseline_p90):
    """
    Genera mean, std, p95 y exceedance_count para cada sensor en cada ventana.
    También genera ratio vs baseline para cada sensor.
    
    - mean/std: nivel y variabilidad (mismos que v05)
    - p95: percentil 95 en ventana — captura picos extremos
    - exceedance: nº de pasos de 10min por encima del p90 del baseline
    - baseline_ratio: media en ventana 7d / media baseline (solo para 7d)
    """
    feats = {}
    for col in sensors:
        if col not in df.columns:
            print(f'  WARN: {col} no encontrada, saltando')
            continue
        s = df[col].ffill().fillna(0)
        
        # Umbral para exceedance (p90 del baseline de operación limpia)
        thresh = baseline_p90.get(col, s.quantile(0.90))
        
        for wname, w in windows.items():
            min_p = max(1, w // 3)
            roll = s.rolling(w, min_periods=min_p)
            
            feats[f'{col}__mean_{wname}']  = roll.mean()
            feats[f'{col}__std_{wname}']   = roll.std().fillna(0)
            feats[f'{col}__p95_{wname}']   = roll.quantile(0.95)
            # Exceedance: fracción de pasos en ventana que superan el umbral baseline
            feats[f'{col}__exceed_{wname}']= s.rolling(w, min_periods=min_p).apply(
                lambda x: (x > thresh).mean() if len(x) > 0 else 0.0,
                raw=True
            )
        
        # Ratio vs baseline (solo ventana 7d — tendencia larga)
        bm = baseline_mean.get(col, 1.0)
        if abs(bm) > 1e-6:
            roll_7d = s.rolling(WINDOWS['7d'], min_periods=max(1, WINDOWS['7d'] // 3))
            feats[f'{col}__baseline_ratio'] = roll_7d.mean() / abs(bm)
        
    return pd.DataFrame(feats, index=df.index)

print('Función make_rolling_features_v2 definida')
print('Stats por ventana: mean, std, p95, exceedance + baseline_ratio(7d)')
print(f'Features por sensor: {len(WINDOWS) * 4 + 1} = {len(WINDOWS)} ventanas × 4 stats + 1 ratio baseline')


Función make_rolling_features_v2 definida
Stats por ventana: mean, std, p95, exceedance + baseline_ratio(7d)
Features por sensor: 17 = 4 ventanas × 4 stats + 1 ratio baseline


---

## 5. Sensores por Familia

Mismos sensores que en la versión 05 — la selección por causalidad física no cambia.


In [11]:
# ==============================================================================
# 5. SENSORES POR FAMILIA (sin cambios respecto a 05)
# ==============================================================================
FAMILY_SENSORS = {

    'yaw_cable': [
        'nacelle_position',
        'nacelle_position_standard_deviation',
        'wind_direction',
        'wind_direction_standard_deviation',
        'vane_position_12',
        'cable_windings_from_calibration_point',
        'wind_speed_ms',
        'power_kw',
        'yaw_error',
        'yaw_error_wind',
        'cable_rate',
        'nacelle_std_ratio',
    ],

    'brake_hydro': [
        'gear_oil_inlet_pressure_bar',
        'gear_oil_pump_pressure_bar',
        'gear_oil_inlet_temperature_c',
        'gear_oil_temperature_c',
        'generator_rpm_rpm',
        'generator_rpm_standard_deviation_rpm',
        'rotor_speed_rpm',
        'power_kw',
        'front_bearing_temperature_c',
        'rear_bearing_temperature_c',
        'metal_particle_count',
        't_gear_oil_delta',
        'pressure_vs_temp',
        'metal_particle_rate',
    ],

    'generator': [
        'generator_bearing_front_temperature_c',
        'generator_bearing_rear_temperature_c',
        'generator_bearing_front_temperature_max_c',
        'generator_bearing_rear_temperature_max_c',
        'nacelle_temperature_c',
        'nacelle_ambient_temperature_c',
        'ambient_temperature_converter_c',
        'power_kw',
        'reactive_power_kvar',
        'power_factor_cosphi',
        'stator_temperature_1_c',
        'wind_speed_ms',
        't_bearing_delta',
        't_rear_bearing_delta',
        't_stator_delta',
        't_bearing_diff',
        't_stator_bearing_diff',
        'apparent_power_kva',
        'reactive_power_ratio',
        't_bearing_delta_roc',
        't_stator_roc',
    ],

    'pitch_bat': [
        'motor_current_axis_1_a',
        'motor_current_axis_2_a',
        'motor_current_axis_3_a',
        'blade_angle_pitch_position_a',
        'blade_angle_pitch_position_b',
        'blade_angle_pitch_position_c',
        'temperature_motor_axis_1_c',
        'temperature_motor_axis_2_c',
        'temperature_motor_axis_3_c',
        'nacelle_ambient_temperature_c',
        'power_kw',
        'wind_speed_ms',
        'pitch_asymmetry',
        'blade_angle_mean',
        'motor_current_imbalance',
    ],
}

for fam, sensors in FAMILY_SENSORS.items():
    n_feat = len(sensors) * (len(WINDOWS) * 4 + 1)
    print(f'{fam:<15}: {len(sensors)} sensores → ~{n_feat} features')


yaw_cable      : 12 sensores → ~204 features
brake_hydro    : 14 sensores → ~238 features
generator      : 21 sensores → ~357 features
pitch_bat      : 15 sensores → ~255 features


---

## 6. Generación y Guardado de Features

⚠️ **Este proceso tarda ~10-20 minutos** por la ventana de 7 días (1008 pasos).  
Los archivos se guardan en `data/silver/features_{familia}_v2.parquet` para no sobreescribir los de v05.


In [12]:
# ==============================================================================
# 6. GENERAR Y GUARDAR FEATURES POR FAMILIA
# ==============================================================================
import time

for family, sensors in FAMILY_SENSORS.items():
    t0 = time.time()
    print(f'\n{"="*70}')
    print(f'Generando features v2 para: {family}')
    print(f'{"="*70}')

    feats = make_rolling_features_v2(df, sensors, WINDOWS, baseline_mean, baseline_p90)

    target_cols = [f'hours_to_{family}', f'is_pre_{family}']
    output = pd.concat([df[['timestamp']], df[target_cols], feats], axis=1)

    # Guardar como _v2 para no sobreescribir los originales
    output_path = os.path.join(silver_dir, f'features_{family}_v2.parquet')
    output.to_parquet(output_path, index=False)

    elapsed = time.time() - t0
    print(f'  Guardado: features_{family}_v2.parquet')
    print(f'  Shape: {output.shape}')
    print(f'  Features: {feats.shape[1]}')
    print(f'  Positivos: {output[f"is_pre_{family}"].sum():,} ({100*output[f"is_pre_{family}"].mean():.1f}%)')
    print(f'  NaN total: {feats.isna().sum().sum():,} ({100*feats.isna().mean().mean():.1f}%)')
    print(f'  Tiempo: {elapsed:.0f}s')

print(f'\n{"="*70}')
print('FEATURE ENGINEERING v2 COMPLETADO')
print('Archivos generados: features_*_v2.parquet')
print(f'{"="*70}')



Generando features v2 para: yaw_cable
  Guardado: features_yaw_cable_v2.parquet
  Shape: (210384, 207)
  Features: 204
  Positivos: 100,065 (47.6%)
  NaN total: 18,204 (0.0%)
  Tiempo: 34s

Generando features v2 para: brake_hydro
  Guardado: features_brake_hydro_v2.parquet
  Shape: (210384, 241)
  Features: 238
  Positivos: 21,185 (10.1%)
  NaN total: 21,238 (0.0%)
  Tiempo: 39s

Generando features v2 para: generator
  Guardado: features_generator_v2.parquet
  Shape: (210384, 360)
  Features: 357
  Positivos: 22,272 (10.6%)
  NaN total: 31,857 (0.0%)
  Tiempo: 58s

Generando features v2 para: pitch_bat
  Guardado: features_pitch_bat_v2.parquet
  Shape: (210384, 258)
  Features: 255
  Positivos: 28,753 (13.7%)
  NaN total: 22,755 (0.0%)
  Tiempo: 42s

FEATURE ENGINEERING v2 COMPLETADO
Archivos generados: features_*_v2.parquet


---

## 7. Verificación de Señal — AUC Univariante v2 vs v05

Comprobamos que las nuevas features tienen mejor AUC que las `slope_*` eliminadas.  
Objetivo: AUC mediana > 0.55 (vs 0.51 en v05).


In [13]:
# ==============================================================================
# 7. VERIFICACIÓN — AUC UNIVARIANTE
# ==============================================================================
from sklearn.metrics import roc_auc_score

print('Comparación AUC univariante: v05 (slope incluido) vs v05b (slope eliminado)\n')
print(f'{"Familia":<15} {"AUC med v05b":>14} {"Top feature":>40} {"AUC top":>8}')
print('-' * 85)

for fam in FAMILY_SENSORS.keys():
    path_v2 = os.path.join(silver_dir, f'features_{fam}_v2.parquet')
    feat = pd.read_parquet(path_v2)
    target_col = f'is_pre_{fam}'
    feat_cols = [c for c in feat.columns if c not in ['timestamp', target_col, f'hours_to_{fam}']]
    
    y = feat[target_col].astype(int)
    aucs = {}
    for col in feat_cols:
        x = feat[col].fillna(0)
        try:
            auc = roc_auc_score(y, x)
            aucs[col] = max(auc, 1 - auc)
        except:
            pass
    
    med = np.median(list(aucs.values()))
    top_feat, top_auc = max(aucs.items(), key=lambda x: x[1])
    print(f'{fam:<15} {med:>14.3f} {top_feat:>40} {top_auc:>8.3f}')

print()
print('Referencia v05: AUC medianas eran 0.510–0.534')
print('Si v05b mejora a >0.540 en todas las familias → las nuevas features aportan señal real')


Comparación AUC univariante: v05 (slope incluido) vs v05b (slope eliminado)

Familia           AUC med v05b                              Top feature  AUC top
-------------------------------------------------------------------------------------
yaw_cable                0.527 nacelle_position_standard_deviation__mean_7d    0.606
brake_hydro              0.534    gear_oil_pump_pressure_bar__exceed_7d    0.608
generator                0.537 generator_bearing_front_temperature_max_c__std_7d    0.670
pitch_bat                0.514                 blade_angle_mean__p95_7d    0.601

Referencia v05: AUC medianas eran 0.510–0.534
Si v05b mejora a >0.540 en todas las familias → las nuevas features aportan señal real


---

## 📋 Resumen de Cambios v05b

| Aspecto | v05 | v05b |
|---------|-----|------|
| Features eliminadas | — | `slope_*` (AUC=0.50, ruido) |
| Features añadidas | — | `p95_*`, `exceed_*`, `baseline_ratio` |
| Stats por ventana | mean + std + slope | mean + std + p95 + exceedance |
| Baseline ratio | No | Sí — ratio vs primeros 180 días |
| Archivos output | `features_{fam}.parquet` | `features_{fam}_v2.parquet` |

### Siguiente paso

**Notebook `06b_training.ipynb`** — entrena los 4 modelos con las features v2,  
threshold ajustado a 0.30 (prioritiza Recall), y comparativa directa con resultados del 06 original.
